# Data Flow Analysis: RMM Action Recognition Pipeline

This notebook tracks data flow through each processing stage, documenting where data is dropped and why.

## Processing Stages

1. **Splits Generation** (this section)
   - Raw annotations → Segments extraction → Train/Val/Test splits
   
2. **Pre-Processing** (to be added)
   - Raw Clips → SAM3 tracked/cropped clips
   - SAM3 clips → HRNet keypoint data
   - Keypoint data → PKL files for skeleton models
   
3. **Data Validation via Model Training** (to be added)
   - SAM3 clips → V-JEPA2 usable data
   - PKL data → Skeleton model usable data


## Stage 1: Splits Generation

**Source Notebook:** `dataprep/clip_gen/rmm_annot.ipynb`

**Output Location:** `dataprep/splits/`

### Overview

The splits generation process extracts RMM segments from raw Excel annotations and creates train/val/test splits respecting LCTO (Leave-Cohort-Timepoint-Out) grouping.


In [ ]:
import pandas as pd
import json
import os
from pathlib import Path

# Set paths
base_dir = Path('/orcd/data/satra/001/users/brukew/actreg/dataprep')
splits_dir = base_dir / 'splits'

print(f"Base directory: {base_dir}")
print(f"Splits directory: {splits_dir}")


### Step 1: Load Split Metadata


In [ ]:
# Load metadata for all split types
cv_5class_meta = json.load(open(splits_dir / 'cv_splits' / 'split_metadata.json'))
cv_4class_meta = json.load(open(splits_dir / 'cv_splits_4class' / 'split_metadata.json'))
single_5class_meta = json.load(open(splits_dir / 'single_split' / 'split_info.json'))
single_4class_meta = json.load(open(splits_dir / 'single_split_4class' / 'split_info.json'))

print("✓ Loaded all split metadata files")
print(f"  5-class CV: {cv_5class_meta['total_segments']} segments")
print(f"  4-class CV: {cv_4class_meta['total_segments']} segments")
print(f"  5-class single: {single_5class_meta['train_segments'] + single_5class_meta['val_segments'] + single_5class_meta['test_segments']} segments")
print(f"  4-class single: {single_4class_meta['train_segments'] + single_4class_meta['val_segments'] + single_4class_meta['test_segments']} segments")


### Step 2: Analyze 5-Class CV Splits


In [ ]:
print("=" * 80)
print("5-CLASS CROSS-VALIDATION SPLITS")
print("=" * 80)

print(f"\nTotal segments: {cv_5class_meta['total_segments']}")
print(f"Total LCTO groups: {cv_5class_meta['total_lcto_groups']}")
print(f"Number of folds: {cv_5class_meta['n_folds']}")

print("\nPer-fold breakdown:")
for fold in cv_5class_meta['folds']:
    print(f"\nFold {fold['fold']}:")
    print(f"  Train: {fold['train_segments']} segments from {fold['train_lcto_groups']} LCTO groups")
    print(f"  Val:   {fold['val_segments']} segments from {fold['val_lcto_groups']} LCTO groups")


In [ ]:
# Load actual CSV files to verify counts and analyze class distribution
fold_0_train_5class = pd.read_csv(splits_dir / 'cv_splits' / 'fold_0_train.csv')
fold_0_val_5class = pd.read_csv(splits_dir / 'cv_splits' / 'fold_0_val.csv')

print("Fold 0 - 5-Class Distribution:")
print("\nTrain set:")
train_counts_5class = fold_0_train_5class['rmm_type'].value_counts().sort_index()
print(train_counts_5class)
print("\nVal set:")
val_counts_5class = fold_0_val_5class['rmm_type'].value_counts().sort_index()
print(val_counts_5class)

print(f"\nTotal unique segment IDs: {fold_0_train_5class['segment_id'].nunique() + fold_0_val_5class['segment_id'].nunique()}")
print(f"Expected from metadata: {cv_5class_meta['total_segments']}")
print(f"Match: {fold_0_train_5class['segment_id'].nunique() + fold_0_val_5class['segment_id'].nunique() == cv_5class_meta['total_segments']}")


### Step 3: Analyze 4-Class CV Splits

**Key Difference:** The 4-class version merges `one hand flap` segments into `hands flapping` (does not filter them out). Total segment count remains 654.


In [ ]:
print("=" * 80)
print("4-CLASS CROSS-VALIDATION SPLITS")
print("=" * 80)

print(f"\nTotal segments: {cv_4class_meta['total_segments']}")
print(f"Total LCTO groups: {cv_4class_meta['total_lcto_groups']}")
print(f"Number of folds: {cv_4class_meta['n_folds']}")
print(f"Classes: {cv_4class_meta['classes']}")

print("\nPer-fold breakdown:")
for fold in cv_4class_meta['folds']:
    print(f"\nFold {fold['fold']}:")
    print(f"  Train: {fold['train_segments']} segments from {fold['train_lcto_groups']} LCTO groups")
    print(f"  Val:   {fold['val_segments']} segments from {fold['val_lcto_groups']} LCTO groups")


In [ ]:
# Load 4-class CSV files
fold_0_train_4class = pd.read_csv(splits_dir / 'cv_splits_4class' / 'fold_0_train.csv')
fold_0_val_4class = pd.read_csv(splits_dir / 'cv_splits_4class' / 'fold_0_val.csv')

print("Fold 0 - 4-Class Distribution:")
print("\nTrain set:")
train_counts_4class = fold_0_train_4class['rmm_type'].value_counts().sort_index()
print(train_counts_4class)
print("\nVal set:")
val_counts_4class = fold_0_val_4class['rmm_type'].value_counts().sort_index()
print(val_counts_4class)

# Compare segment counts
print(f"\n5-class train segments: {len(fold_0_train_5class)}")
print(f"4-class train segments: {len(fold_0_train_4class)}")
print(f"Difference: {len(fold_0_train_5class) - len(fold_0_train_4class)} segments")

# Check if "one hand flap" was filtered or merged
ohf_5class = (fold_0_train_5class['rmm_type'] == 'one hand flap').sum()
ohf_4class = (fold_0_train_4class['rmm_type'] == 'one hand flap').sum()
hf_5class = (fold_0_train_5class['rmm_type'] == 'hands flapping').sum()
hf_4class = (fold_0_train_4class['rmm_type'] == 'hands flapping').sum()

print(f"\n'one hand flap' in 5-class train: {ohf_5class}")
print(f"'one hand flap' in 4-class train: {ohf_4class}")
print(f"'hands flapping' in 5-class train: {hf_5class}")
print(f"'hands flapping' in 4-class train: {hf_4class}")
print(f"Expected merged count: {hf_5class + ohf_5class}")
print(f"Actual 4-class count: {hf_4class}")
print(f"Match: {hf_4class == hf_5class + ohf_5class}")


### Step 4: Analyze Single Splits


In [ ]:
print("=" * 80)
print("SINGLE TRAIN/VAL/TEST SPLITS")
print("=" * 80)

print("\n5-Class Single Split:")
print(f"  Train: {single_5class_meta['train_segments']} segments from {single_5class_meta['train_lcto_groups']} LCTO groups")
print(f"  Val:   {single_5class_meta['val_segments']} segments from {single_5class_meta['val_lcto_groups']} LCTO groups")
print(f"  Test:  {single_5class_meta['test_segments']} segments from {single_5class_meta['test_lcto_groups']} LCTO groups")
total_5class = single_5class_meta['train_segments'] + single_5class_meta['val_segments'] + single_5class_meta['test_segments']
print(f"  Total: {total_5class} segments")

print("\n4-Class Single Split:")
print(f"  Train: {single_4class_meta['train_segments']} segments from {single_4class_meta['train_lcto_groups']} LCTO groups")
print(f"  Val:   {single_4class_meta['val_segments']} segments from {single_4class_meta['val_lcto_groups']} LCTO groups")
print(f"  Test:  {single_4class_meta['test_segments']} segments from {single_4class_meta['test_lcto_groups']} LCTO groups")
total_4class = single_4class_meta['train_segments'] + single_4class_meta['val_segments'] + single_4class_meta['test_segments']
print(f"  Total: {total_4class} segments")

print(f"\nTotal segments match: {total_5class == total_4class == cv_5class_meta['total_segments']}")


### Step 5: Data Flow Summary

**Key Observations:**

1. **Total Segments:** 654 segments across all splits
2. **LCTO Groups:** 151 unique (child, timepoint) combinations
3. **5-Class vs 4-Class:** The 4-class variant merges `one hand flap` into `hands flapping` (does not filter them out)
4. **Split Consistency:** CV splits maintain same segment counts across folds (436 train, 218 val per fold)
5. **Source:** Segments extracted from Excel annotations (`RMM_with_ELE_Highlighted.xlsx`) via `rmm_annot.ipynb`


In [ ]:
# Create summary dataframe
summary_data = {
    'Split Type': ['5-Class CV', '4-Class CV', '5-Class Single', '4-Class Single'],
    'Total Segments': [
        cv_5class_meta['total_segments'],
        cv_4class_meta['total_segments'],
        total_5class,
        total_4class
    ],
    'LCTO Groups': [
        cv_5class_meta['total_lcto_groups'],
        cv_4class_meta['total_lcto_groups'],
        single_5class_meta['train_lcto_groups'] + single_5class_meta['val_lcto_groups'] + single_5class_meta['test_lcto_groups'],
        single_4class_meta['train_lcto_groups'] + single_4class_meta['val_lcto_groups'] + single_4class_meta['test_lcto_groups']
    ],
    'Classes': [
        '5 (hands flapping, one hand flap, jumping, rocking, spinning)',
        '4 (hands flapping, jumping, rocking, spinning)',
        '5 (hands flapping, one hand flap, jumping, rocking, spinning)',
        '4 (hands flapping, jumping, rocking, spinning)'
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\n" + "=" * 80)
print("SPLITS SUMMARY")
print("=" * 80)
print(summary_df.to_string(index=False))


### Step 6: Verify Data Integrity

Check for:
- Segment count consistency
- LCTO group overlap between splits
- Class distribution balance


In [ ]:
# Verify no overlap in LCTO groups between train/val in CV splits
print("=" * 80)
print("DATA INTEGRITY CHECKS")
print("=" * 80)

for fold_idx, fold in enumerate(cv_5class_meta['folds']):
    train_lcto = set(fold['train_lcto_group_list'])
    val_lcto = set(fold['val_lcto_group_list'])
    overlap = train_lcto & val_lcto
    print(f"\nFold {fold_idx} (5-class):")
    print(f"  Train LCTO groups: {len(train_lcto)}")
    print(f"  Val LCTO groups: {len(val_lcto)}")
    print(f"  Overlap: {len(overlap)} groups")
    if overlap:
        print(f"  ⚠️  WARNING: Overlap detected: {overlap}")
    else:
        print(f"  ✓ No overlap detected")

# Check segment ID uniqueness
all_segment_ids_5class = set()
for fold in cv_5class_meta['folds']:
    train_df = pd.read_csv(splits_dir / 'cv_splits' / f"fold_{fold['fold']}_train.csv")
    val_df = pd.read_csv(splits_dir / 'cv_splits' / f"fold_{fold['fold']}_val.csv")
    all_segment_ids_5class.update(train_df['segment_id'].tolist())
    all_segment_ids_5class.update(val_df['segment_id'].tolist())

print(f"\nTotal unique segment IDs across all CV folds (5-class): {len(all_segment_ids_5class)}")
print(f"Expected: {cv_5class_meta['total_segments']}")
print(f"Match: {len(all_segment_ids_5class) == cv_5class_meta['total_segments']}")


## Stage 2: Clip Creation

**Source Script:** `dataprep/clip_gen/create_clip_segments.py`  
**Output Location:** `/orcd/scratch/bcs/001/sensein/sails/rmm/classification_clips/`

### Overview

The clip creation process extracts video segments from the full-length standardized videos based on timestamps in the split CSVs. Each segment is cut using ffmpeg and validated for proper encoding.

**Key Features:**
- Reads CSV files from splits directory (default: `dataprep/splits/cv_splits/`)
- Cuts clips using `start_sec` and `end_sec` timestamps
- Creates canonical clips (deduplicated) and links them to fold-specific directories
- Validates clips using ffprobe and decord (if available)
- Generates a manifest CSV tracking all created clips


In [ ]:
# Set paths for clip creation analysis
clips_root = Path('/orcd/scratch/bcs/001/sensein/sails/rmm/classification_clips')
canonical_clips_dir = clips_root / 'canonical_clips'
manifest_path = clips_root / 'clip_manifest.csv'

print(f"Clips root: {clips_root}")
print(f"Canonical clips directory: {canonical_clips_dir}")
print(f"Manifest path: {manifest_path}")
print(f"\nCanonical clips directory exists: {canonical_clips_dir.exists()}")
print(f"Manifest exists: {manifest_path.exists()}")


### Step 1: Load Clip Manifest and Analyze Created Clips


In [ ]:
# Load manifest if it exists
if manifest_path.exists():
    manifest_df = pd.read_csv(manifest_path)
    print(f"Manifest loaded: {len(manifest_df)} rows")
    print(f"\nManifest columns: {list(manifest_df.columns)}")
    print(f"\nUnique segment IDs in manifest: {manifest_df['segment_id'].nunique()}")
    print(f"Unique CSV files processed: {manifest_df['csv_file'].nunique()}")
    print(f"\nCSV files in manifest:")
    print(manifest_df['csv_file'].value_counts().sort_index())
else:
    print(f"⚠️  Manifest not found at {manifest_path}")
    manifest_df = None


In [ ]:
# Count clips in canonical_clips directory
if canonical_clips_dir.exists():
    canonical_clips = list(canonical_clips_dir.glob('*.mp4'))
    print(f"Canonical clips directory: {canonical_clips_dir}")
    print(f"Number of .mp4 files: {len(canonical_clips)}")
    
    # Extract segment IDs from filenames
    canonical_segment_ids = {f.stem for f in canonical_clips}
    print(f"Unique segment IDs: {len(canonical_segment_ids)}")
    
    # Check for any non-mp4 files
    other_files = [f for f in canonical_clips_dir.iterdir() if f.is_file() and not f.name.endswith('.mp4')]
    if other_files:
        print(f"\n⚠️  Non-mp4 files found: {len(other_files)}")
        for f in other_files[:5]:
            print(f"  {f.name}")
else:
    print(f"⚠️  Canonical clips directory not found: {canonical_clips_dir}")
    canonical_segment_ids = set()


### Step 2: Compare Expected vs Actual Clips

Compare segments from splits with clips that were actually created.


In [ ]:
# Get all segment IDs from splits
all_split_segment_ids = set()
for fold in cv_5class_meta['folds']:
    train_df = pd.read_csv(splits_dir / 'cv_splits' / f"fold_{fold['fold']}_train.csv")
    val_df = pd.read_csv(splits_dir / 'cv_splits' / f"fold_{fold['fold']}_val.csv")
    all_split_segment_ids.update(train_df['segment_id'].tolist())
    all_split_segment_ids.update(val_df['segment_id'].tolist())

print(f"Total segments in splits: {len(all_split_segment_ids)}")
print(f"Expected from metadata: {cv_5class_meta['total_segments']}")
print(f"Match: {len(all_split_segment_ids) == cv_5class_meta['total_segments']}")

if manifest_df is not None:
    manifest_segment_ids = set(manifest_df['segment_id'].unique())
    print(f"\nSegments in manifest: {len(manifest_segment_ids)}")
    
    missing_in_manifest = all_split_segment_ids - manifest_segment_ids
    extra_in_manifest = manifest_segment_ids - all_split_segment_ids
    
    print(f"\nMissing in manifest: {len(missing_in_manifest)}")
    if missing_in_manifest:
        print(f"  Examples: {list(missing_in_manifest)[:5]}")
    print(f"Extra in manifest: {len(extra_in_manifest)}")
    if extra_in_manifest:
        print(f"  Examples: {list(extra_in_manifest)[:5]}")

if canonical_segment_ids:
    print(f"\nCanonical clips created: {len(canonical_segment_ids)}")
    
    missing_clips = all_split_segment_ids - canonical_segment_ids
    extra_clips = canonical_segment_ids - all_split_segment_ids
    
    print(f"\nMissing clips: {len(missing_clips)}")
    if missing_clips:
        print(f"  Examples: {list(missing_clips)[:5]}")
    print(f"Extra clips: {len(extra_clips)}")
    if extra_clips:
        print(f"  Examples: {list(extra_clips)[:5]}")


### Step 3: Validate Clip Encoding

Verify that clips are properly encoded and can be read by video loaders (ffprobe and decord).


In [ ]:
import subprocess
from pathlib import Path

def clip_is_valid(path: Path) -> tuple[bool, str]:
    """
    Validate a clip using ffprobe (same logic as create_clip_segments.py).
    Returns (is_valid, error_message).
    """
    try:
        if not path.exists():
            return False, "File does not exist"
        
        # Check if symlink target exists
        if path.is_symlink():
            try:
                _ = path.resolve(strict=True)
            except FileNotFoundError:
                return False, "Broken symlink"
        
        # Check file size
        if path.stat().st_size == 0:
            return False, "File is empty"
        
        # Use ffprobe to check if video is readable
        probe_cmd = [
            "ffprobe",
            "-v", "error",
            "-select_streams", "v:0",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            str(path),
        ]
        result = subprocess.run(probe_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=10)
        if result.returncode != 0:
            return False, "ffprobe failed"
        
        return True, ""
    except subprocess.TimeoutExpired:
        return False, "ffprobe timeout"
    except Exception as e:
        return False, f"Error: {str(e)}"

print("Clip validation function defined.")


In [ ]:
# Validate a sample of clips
if canonical_segment_ids:
    sample_size = 50 #len(canonical_segment_ids)
    sample_clips = list(canonical_segment_ids)[:sample_size]
    
    print(f"Validating {sample_size} sample clips...")
    validation_results = []
    
    for segment_id in sample_clips:
        clip_path = canonical_clips_dir / f"{segment_id}.mp4"
        is_valid, error = clip_is_valid(clip_path)
        validation_results.append({
            'segment_id': segment_id,
            'is_valid': is_valid,
            'error': error,
            'exists': clip_path.exists()
        })
    
    validation_df = pd.DataFrame(validation_results)
    
    valid_count = validation_df['is_valid'].sum()
    invalid_count = (~validation_df['is_valid']).sum()
    
    print(f"\nValidation Results (sample of {sample_size}):")
    print(f"  Valid: {valid_count} ({100*valid_count/sample_size:.1f}%)")
    print(f"  Invalid: {invalid_count} ({100*invalid_count/sample_size:.1f}%)")
    
    if invalid_count > 0:
        print(f"\nInvalid clips:")
        invalid_df = validation_df[~validation_df['is_valid']]
        for _, row in invalid_df.iterrows():
            print(f"  {row['segment_id']}: {row['error']}")
else:
    print("No canonical clips found to validate.")


### Step 4: Analyze Clip Duration and Distribution

Check clip durations and class distribution in created clips.


In [ ]:
if manifest_df is not None:
    print("=" * 80)
    print("CLIP DURATION ANALYSIS")
    print("=" * 80)
    
    # Convert duration to float
    manifest_df['duration_float'] = pd.to_numeric(manifest_df['duration'], errors='coerce')
    
    print(f"\nDuration statistics:")
    print(f"  Mean: {manifest_df['duration_float'].mean():.2f} seconds")
    print(f"  Median: {manifest_df['duration_float'].median():.2f} seconds")
    print(f"  Min: {manifest_df['duration_float'].min():.2f} seconds")
    print(f"  Max: {manifest_df['duration_float'].max():.2f} seconds")
    print(f"  Std: {manifest_df['duration_float'].std():.2f} seconds")
    
    # Check for zero-length or very short clips
    short_clips = manifest_df[manifest_df['duration_float'] < 0.5]
    print(f"\nClips shorter than 0.5 seconds: {len(short_clips)}")
    if len(short_clips) > 0:
        print("  Examples:")
        for _, row in short_clips.head(5).iterrows():
            print(f"    {row['segment_id']}: {row['duration_float']:.3f}s")
    
    # Class distribution
    if 'label' in manifest_df.columns:
        print(f"\nClass distribution in manifest:")
        print(manifest_df['label'].value_counts().sort_index())
        
        # Duration by class
        print(f"\nDuration statistics by class:")
        if 'label' in manifest_df.columns and 'duration_float' in manifest_df.columns:
            duration_by_class = manifest_df.groupby('label')['duration_float'].agg(['mean', 'median', 'min', 'max', 'count'])
            print(duration_by_class.round(2))
else:
    print("⚠️  Manifest not loaded. Run previous cells to load manifest data.")

### Step 5: Data Flow Summary for Clip Creation

**Key Observations:**
1. **Input:** CSV split files from `dataprep/splits/cv_splits/`
2. **Processing:** ffmpeg-based clip extraction using `start_sec` and `end_sec` timestamps
3. **Output:** Canonical clips in `canonical_clips/` directory, linked to fold-specific subdirectories
4. **Validation:** Clips validated using ffprobe and optionally decord
5. **Manifest:** `clip_manifest.csv` tracks all successfully created clips


In [ ]:
# Create summary
clip_summary = {
    'Stage': 'Clip Creation',
    'Input': 'CSV splits from dataprep/splits/cv_splits/',
    'Expected Segments': cv_5class_meta['total_segments'],
    'Canonical Clips Created': len(canonical_segment_ids) if canonical_segment_ids else 0,
    'Manifest Entries': len(manifest_df) if manifest_df is not None else 0,
    'Unique Segments in Manifest': manifest_df['segment_id'].nunique() if manifest_df is not None else 0,
}

print("=" * 80)
print("CLIP CREATION SUMMARY")
print("=" * 80)
for key, value in clip_summary.items():
    print(f"{key}: {value}")

# Check for missing clips (if variables are defined)
try:
    if canonical_segment_ids and 'all_split_segment_ids' in globals() and all_split_segment_ids:
        missing = len(all_split_segment_ids - canonical_segment_ids)
        print(f"\nMissing clips: {missing}")
        if missing == 0:
            print("✓ All expected clips were created successfully!")
        else:
            print(f"⚠️  {missing} clips are missing")
except NameError:
    print("\n⚠️  Run previous cells to compare expected vs actual clips")


## Stage 3: SAM3 Mask Generation

**Source Script:** `sailsprep/feature_processing/tracker/sam3/run_sam3_batch.py`  
**Batch Script:** `sails_sbatch/tracker/sam3_track.sh`  
**Input CSV:** `sailsprep/subset_data/RMM.csv`  
**Output Location:** `/orcd/scratch/bcs/001/sensein/sails/cache_for_tracking/masks/`

### Overview

The SAM3 mask generation process tracks and segments people in videos using SAM3 (Segment Anything Model 3). Mask caches are stored as HDF5 files containing per-frame masks, bounding boxes, scores, and object IDs.

**Key Features:**
- Processes videos from RMM.csv
- Tracks people using text prompt "person"
- Capped at 5400 frames per video (`--max-frames 5400`)
- Saves mask caches as HDF5 files
- Stores progress in `processing_progress.json`


In [ ]:
# Set paths for SAM3 analysis
rmm_csv_path = Path('/orcd/data/satra/001/users/brukew/sailsprep/subset_data/RMM.csv')
mask_cache_base = Path('/orcd/scratch/bcs/001/sensein/sails/cache_for_tracking/masks')
sam3_output_dir = Path('/orcd/scratch/bcs/001/sensein/sails/feature_processing/sam3_outputs/RMM')
video_meta_path = Path('/orcd/data/satra/001/users/brukew/actreg/dataprep/video_meta.json')
progress_file = sam3_output_dir / 'processing_progress.json'

print(f"RMM CSV: {rmm_csv_path}")
print(f"Mask cache base: {mask_cache_base}")
print(f"SAM3 output dir: {sam3_output_dir}")
print(f"Progress file: {progress_file}")
print(f"\nRMM CSV exists: {rmm_csv_path.exists()}")
print(f"Mask cache base exists: {mask_cache_base.exists()}")
print(f"Progress file exists: {progress_file.exists()}")


### Step 1: Load RMM CSV and Expected Videos


In [ ]:
# Load video metadata JSON
import json
with open(video_meta_path, 'r') as f:
    video_meta_data = json.load(f)

video_meta_records = video_meta_data['records']
print(f"Video metadata loaded: {len(video_meta_records)} records")
print(f"Generated at: {video_meta_data.get('generated_at', 'N/A')}")

# Convert to DataFrame for easier analysis
video_meta_df = pd.DataFrame(video_meta_records)
print(f"\nVideo metadata columns: {list(video_meta_df.columns)}")
print(f"Unique FileNames: {video_meta_df['FileName'].nunique()}")

# Extract cache basename from mask_cache_path
# Path format: /orcd/scratch/.../masks/{basename}/facebook-sam3__prompt-person.h5
video_meta_df['cache_basename'] = video_meta_df['mask_cache_path'].apply(
    lambda p: Path(p).parent.name if pd.notna(p) else None
)

# Extract filename stem (without extension)
video_meta_df['filename_stem'] = video_meta_df['FileName'].apply(lambda x: Path(x).stem if pd.notna(x) else None)

print(f"\nVideos with mask cache paths: {video_meta_df['mask_cache_path'].notna().sum()}")
print(f"\nSample video metadata:")
print(video_meta_df[['FileName', 'filename_stem', 'cache_basename', 'fps', 'duration', 'cache_frames']].head(10))


### Step 3: Check Mask Cache Files


In [ ]:
# Get actual cache files from filesystem (for verification)
mask_cache_pattern = "facebook-sam3__prompt-person.h5"
mask_cache_files = list(mask_cache_base.glob(f"*/{mask_cache_pattern}"))

print(f"Found {len(mask_cache_files)} mask cache files on filesystem")

# Extract video basenames from cache paths
actual_cache_basenames = set()
for cache_file in mask_cache_files:
    # Path structure: masks/{video_basename}/facebook-sam3__prompt-person.h5
    video_basename = cache_file.parent.name
    actual_cache_basenames.add(video_basename)

print(f"Unique video basenames with caches: {len(actual_cache_basenames)}")

# Get expected cache basenames from metadata
expected_cache_basenames_from_meta = set(video_meta_df['cache_basename'].dropna().unique())
print(f"\nExpected cache basenames from metadata: {len(expected_cache_basenames_from_meta)}")
print(f"\nSample cache basenames from metadata:")
print(list(expected_cache_basenames_from_meta)[:10])
print(len(video_meta_df['cache_basename'].dropna()))
video_meta_df[video_meta_df.duplicated(subset=['cache_basename'], keep=False)]["original_path"]

### Step 4: Compare Expected vs Actual Mask Caches


In [ ]:
# Compare expected vs actual
missing_caches = expected_cache_basenames_from_meta - actual_cache_basenames
extra_caches = actual_cache_basenames - expected_cache_basenames_from_meta

print(f"Expected cache basenames (from metadata): {len(expected_cache_basenames_from_meta)}")
print(f"Actual cache basenames (on filesystem): {len(actual_cache_basenames)}")
print(f"\nMissing caches: {len(missing_caches)}")
if missing_caches:
    print(f"  Examples: {list(missing_caches)[:10]}")
print(f"\nExtra caches (not in metadata): {len(extra_caches)}")
if extra_caches:
    print(f"  Examples: {list(extra_caches)[:10]}")

# Check completion status using metadata
video_meta_df['has_cache'] = video_meta_df['cache_basename'].isin(actual_cache_basenames)
video_meta_df['cache_exists'] = video_meta_df['mask_cache_path'].apply(
    lambda p: Path(p).exists() if pd.notna(p) else False
)

print(f"\nVideos with mask caches (from metadata): {video_meta_df['has_cache'].sum()} / {len(video_meta_df)}")
print(f"Videos with cache files that exist: {video_meta_df['cache_exists'].sum()} / {len(video_meta_df)}")


### Step 5: Analyze Frame Counts in Mask Caches

Check how many frames were processed vs total frames per video (capped at 5400).


In [ ]:
import h5py

def analyze_mask_cache(cache_path: Path) -> dict:
    """Analyze a mask cache file to extract frame information."""
    with h5py.File(cache_path, 'r') as f:
        # Get frame indices
        frame_keys = [k for k in f.keys() if k.startswith('frame_')]
        frame_indices = sorted([int(k.split('_')[1]) for k in frame_keys])
        
        # Get attributes
        attrs = dict(f.attrs)
        
        return {
            'num_frames': len(frame_indices),
            'frame_indices': frame_indices,
            'min_frame': int(min(frame_indices)) if frame_indices else None,
            'max_frame': int(max(frame_indices)) if frame_indices else None,
            'frame_stride': int(attrs.get('frame_stride', None)),
            'max_frames': int(attrs.get('max_frames', None)),
            'height': int(attrs.get('height', None)),
            'width': int(attrs.get('width', None)),
            'model': attrs.get('model', None),
            'prompt': attrs.get('prompt', None),
        }

print("Mask cache analysis function defined.")


In [ ]:
analyze_mask_cache("/orcd/scratch/bcs/001/sensein/sails/cache_for_tracking/masks/12995341_1110618565663667_1956232591_n_segmented/facebook-sam3__prompt-person.h5")

In [ ]:
# Analyze frame counts using metadata
cache_issues = []
for _, meta_row in video_meta_df.iterrows():
    cache_path = Path(meta_row['mask_cache_path'])
    if cache_path.exists():
        try:
            cache_info = analyze_mask_cache(cache_path)
            video_meta_df.loc[_, 'cache_frames'] = cache_info['max_frame']
            # if 'error' in cache_info:
            #     print(cac)
            #     print(f"Error analyzing {cache_path}: {cache_info['error']}")
        except Exception as e:
            video_meta_df.loc[_, 'cache_frames'] = None
            video_meta_df.loc[_, 'has_cache'] = False
            cache_issues.append(meta_row['SourceFile'])
            print(f"Error analyzing {cache_path}: {str(e)}")
        
valid_caches = video_meta_df[video_meta_df['cache_frames'].notna()].copy()
print(f"Videos with cache_frames data: {len(valid_caches)}")
print(f"Videos with cache issues: {len(cache_issues)}")
print(f"Cache issues: {cache_issues}")

if len(valid_caches) > 0:
    print(f"\nFrame count statistics (from metadata):")
    print(f"  Mean frames: {valid_caches['cache_frames'].mean():.1f}")
    print(f"  Median frames: {valid_caches['cache_frames'].median():.1f}")
    print(f"  Min frames: {valid_caches['cache_frames'].min()}")
    print(f"  Max frames: {valid_caches['cache_frames'].max()}")
    
    # Check for videos at the 5400 frame cap
    at_cap = valid_caches[valid_caches['cache_frames'] == 5400]
    print(f"\nVideos at 5400 frame cap: {len(at_cap)} ({100*len(at_cap)/len(valid_caches):.1f}%)")
    
    # Check frame stride (from metadata)
    if 'cache_stride' in valid_caches.columns:
        stride_counts = valid_caches['cache_stride'].value_counts()
        print(f"\nFrame stride distribution:")
        print(stride_counts)
    
    # Show examples of videos at cap
    if len(at_cap) > 0:
        print(f"\nSample videos at 5400 frame cap:")
        print(at_cap[['FileName', 'cache_basename', 'cache_frames', 'fps', 'duration']].head(10))
    
    # Store for use in next cell
    cache_analysis_df = valid_caches.copy()
    cache_analysis_df['num_frames'] = cache_analysis_df['cache_frames']
    cache_analysis_df['video_basename'] = cache_analysis_df['cache_basename']
else:
    print("⚠️  No valid cache data found in metadata")

In [ ]:
video_meta_df

In [ ]:
vid_mask_info = []
for _, meta_row in video_meta_df.iterrows():
    mask_info = {}
    source_file = meta_row['SourceFile']
    cache_basename = meta_row['cache_basename']
    mask_info["row_idx"] = meta_row['row_idx']
    mask_info["source_file"] = source_file
    cache_path = meta_row['mask_cache_path']
    mask_info["cache_path"] = cache_path
    if Path(cache_path).exists():
        cache_info = analyze_mask_cache(Path(cache_path))
        mask_info['cache_info'] = cache_info
        # print(f"Saved mask info for {meta_row['row_idx']}: {source_file}")
    else:
        print(f"  Cache does not exist for {source_file}")
        mask_info['cache_info'] = {}
    vid_mask_info.append(mask_info)

vid_mask_info_df = pd.DataFrame(vid_mask_info)
vid_mask_info_df


In [ ]:
import time

video_mask_path = Path('/orcd/data/satra/001/users/brukew/actreg/dataprep/video_mask_info.json')
video_mask_data = {"generated_at": time.strftime("%Y-%m-%dT%H:%M:%S.%fZ"), "count": len(vid_mask_info), "records": vid_mask_info}

with open(video_mask_path, 'w') as f:
    json.dump(video_mask_data, f, indent=2)

### Step 6: Check Frame Cap Impact on Clips

Analyze if videos capped at certain amount of frames were cut off before or during clips defined in the splits.



In [ ]:
# Load clips from splits to get clip timestamps
video_root = Path('/orcd/data/satra/002/datasets/SAILS/Phase_III_Videos/Videos_from_external_standardized')
video_meta = Path('/orcd/data/satra/002/datasets/SAILS/Phase_III_Videos/video_meta.json')

# Get all clips from splits
all_clips = []
for fold in cv_5class_meta['folds']:
    train_df = pd.read_csv(splits_dir / 'cv_splits' / f"fold_{fold['fold']}_train.csv")
    val_df = pd.read_csv(splits_dir / 'cv_splits' / f"fold_{fold['fold']}_val.csv")
    all_clips.append(train_df)
    all_clips.append(val_df)
    break

clips_df = pd.concat(all_clips, ignore_index=True)
print(f"Total clips from splits: {len(clips_df)}")
print(f"Unique videos: {clips_df['filename'].nunique()}")

# Group clips by video filename
clips_by_video = clips_df.groupby('filename')
print(f"\nVideos with clips: {len(clips_by_video)}")

clips_df


In [ ]:
affected_clips = []

for _, meta_row in cache_analysis_df.iterrows():
    filename_stem = meta_row['filename_stem']
    fps = meta_row.get('fps')
    
    if pd.isna(fps) or fps <= 0:
        continue
    
    frames = meta_row['cache_frames']

    # Calculate timestamp of last masked frame
    mask_end_sec = frames / fps
    
    # Find clips from this video by matching filename
    # Try multiple matching strategies
    video_clips = clips_df[
    (
        clips_df['filename'].apply(lambda x: Path(x).stem if pd.notna(x) else '') 
        == filename_stem
    )
    & (
        clips_df['child_id'].astype(str).apply(lambda cid: cid in str(meta_row['SourceFile']))
    )
]
    
    # If no match, try case-insensitive contains
    if len(video_clips) == 0:
        video_clips = clips_df[
            (
                clips_df['filename'].str.contains(filename_stem, case=False, na=False, regex=False) 
                )
            & (
            clips_df['child_id'].astype(str).apply(lambda cid: cid in str(meta_row['SourceFile']))
            )
        ]
    
    if len(video_clips) > 0:
        # Check each clip
        for _, clip_row in video_clips.iterrows():
            start_sec = clip_row.get('start_sec')
            end_sec = clip_row.get('end_sec')
            
            if pd.notna(start_sec) and pd.notna(end_sec):
                try:
                    start_sec = float(start_sec)
                    end_sec = float(end_sec)
                    
                    # Check if clip is affected
                    if start_sec >= mask_end_sec + 1: # 1 sec delta to account for rounding errors
                        # Clip starts after frame cap
                        affected_clips.append({
                            'video_basename': meta_row['cache_basename'],
                            'source_file': meta_row['SourceFile'],
                            'FileName': meta_row['FileName'],
                            'segment_id': clip_row.get('segment_id', ''),
                            'start_sec': start_sec,
                            'end_sec': end_sec,
                            'mask_end_sec': mask_end_sec,
                            'fps': fps,
                            'height': meta_row.get('height'),
                            'width': meta_row.get('width'),
                            'cached_frames': frames,
                            'actual_frames': fps * meta_row.get('duration'),
                            'duration': meta_row.get('duration'),
                            'status': 'clip_starts_after_cap',
                        })
                    elif end_sec > mask_end_sec + 1: # 1 sec delta to account for rounding errors
                        # Clip overlaps with frame cap
                        affected_clips.append({
                            'video_basename': meta_row['cache_basename'],
                            'source_file': meta_row['SourceFile'],
                            'FileName': meta_row['FileName'],
                            'segment_id': clip_row.get('segment_id', ''),
                            'start_sec': start_sec,
                            'end_sec': end_sec,
                            'mask_end_sec': mask_end_sec,
                            'fps': fps,
                            'height': meta_row.get('height'),
                            'width': meta_row.get('width'),
                            'cached_frames': frames,
                            'actual_frames': fps * meta_row.get('duration'),
                            'duration': meta_row.get('duration'),
                            'status': 'clip_overlaps_cap',
                        })
                except (ValueError, TypeError):
                    continue

if affected_clips:
    affected_df = pd.DataFrame(affected_clips)
    print(f"\n⚠️  Found {len(affected_clips)} clips affected by frame cap:")
    print(f"\n  Clips starting after cap: {len(affected_df[affected_df['status'] == 'clip_starts_after_cap'])}")
    print(f"  Clips overlapping cap: {len(affected_df[affected_df['status'] == 'clip_overlaps_cap'])}")
    
    # print(f"\nSample affected clips:")
    # print(affected_df[['FileName', 'segment_id', 'start_sec', 'end_sec', 'mask_end_sec', 'fps', 'status']].head(10))
else:
    print(f"\n✓ No clips affected by frame cap (all clips start before frame cap0)")

affected_clips

In [ ]:
affected_videos = {}
for _, clip_row in affected_df.iterrows():
    if clip_row['source_file'] not in affected_videos:
        affected_videos[clip_row['source_file']] = {
            'clips': [clip_row['segment_id']],
            'num_clips': 1,
            'source_file': clip_row['source_file'],
            'fps': clip_row['fps'],
            'height': clip_row['height'],
            'width': clip_row['width'],
            'duration': clip_row['duration'],
            'mask_end_sec': clip_row['mask_end_sec'],
            'min_end_sec_needed': clip_row['end_sec'],
            'cached_frames': clip_row['cached_frames'],
            'min_frame_needed': clip_row['end_sec']*clip_row['fps'],
        }
    else:
        affected_videos[clip_row['source_file']]['clips'].append(clip_row['segment_id'])
        affected_videos[clip_row['source_file']]['num_clips'] += 1
        if clip_row['end_sec'] > affected_videos[clip_row['source_file']]['min_end_sec_needed']:
            affected_videos[clip_row['source_file']]['min_end_sec_needed'] = clip_row['end_sec']
            affected_videos[clip_row['source_file']]['min_frame_needed'] = clip_row['end_sec']*clip_row['fps']

if affected_videos:
    affected_videos_df = pd.DataFrame(affected_videos.values())
    # print(f"\n⚠️  Found {len(affected_videos_df)} videos affected by frame cap:")
    # print(affected_videos_df[['source_file', 'clips', 'min_end_sec_needed', 'min_frame_needed', 'cached_frames']].head(10))

affected_videos_df


In [ ]:
# save to csv
# affected_videos_df.to_csv('affected_videos.csv', index=False)


### Step 7: SAM3 Mask Generation Summary

**Key Observations:**
1. **Input:** Videos from `RMM.csv` (374 videos)
2. **Processing:** SAM3 tracking with "person" prompt, capped at 5400 frames
3. **Output:** HDF5 mask cache files in `cache_for_tracking/masks/`
4. **Progress Tracking:** `processing_progress.json` tracks completed videos
5. **Frame Limiting:** Videos longer than 5400 frames are truncated


In [ ]:
# Create summary
sam3_summary = {
    'Stage': 'SAM3 Mask Generation',
    'Input Metadata': 'actreg/dataprep/video_meta.json',
    'Total Videos in Metadata': len(video_meta_df),
    'Videos with Mask Caches': video_meta_df['has_cache'].sum(),
    'Videos with Cache Issues': len(cache_issues),
    'Cache Issues': cache_issues,
    'Videos with Cache Files Existing': video_meta_df['cache_exists'].sum(),
    'Total Mask Cache Files (on filesystem)': len(mask_cache_files),
    'Max Frames Cap': 5400,
}

print("=" * 80)
print("SAM3 MASK GENERATION SUMMARY")
print("=" * 80)
for key, value in sam3_summary.items():
    print(f"{key}: {value}")

missing_count = len(missing_caches) 
issues_count = len(cache_issues)
if missing_count > 0:
    print(f"\n⚠️  {missing_count} videos missing mask caches")
else:
    if issues_count > 0:
        print(f"\n⚠️  {issues_count} videos have cache issues")
    else:
        print(f"\n✓ All expected videos have mask caches")


## Stage 2c: HRNet Keypoint Extraction

**Source Script:** `actreg/dataprep/pose_gen/batch_sam_pose.py`  
**Output Location:** `/orcd/scratch/bcs/001/sensein/sails/cache_for_tracking/pose_sam3/`

### Overview

The HRNet keypoint extraction process uses SAM3 bounding boxes to guide MMPose estimation. Pose keypoints are extracted for target child IDs and cached to HDF5 files for downstream processing.

**Key Features:**
- Uses SAM3 bounding boxes to guide pose estimation
- Processes videos from video_meta.json
- Caches pose results to HDF5 files
- Supports resuming from progress file


In [ ]:
# Set paths for pose cache analysis
pose_cache_base = Path('/orcd/scratch/bcs/001/sensein/sails/cache_for_tracking/pose_sam3')

# Pose cache naming convention from batch_sam_pose.py:
# det_name = "dino-5scale_swin-l_8xb2-36e_coco"
# pose_name = "td-hm_hrnet-w48_dark-8xb32-210e_coco-wholebody-384x288"
# det_conf_thresh = 0.5
# filename = f"{det_name}_{det_conf_thresh}_{pose_name}_sam3guided.h5"

det_name = "dino-5scale_swin-l_8xb2-36e_coco"
pose_name = "td-hm_hrnet-w48_dark-8xb32-210e_coco-wholebody-384x288"
det_conf_thresh = 0.5
pose_cache_filename = f"{det_name}_{det_conf_thresh}_{pose_name}_sam3guided.h5"

print(f"Pose cache base: {pose_cache_base}")
print(f"Pose cache filename pattern: {pose_cache_filename}")
print(f"\nPose cache base exists: {pose_cache_base.exists()}")


### Step 1: Check Pose Caches for All Videos

Verify that each video in video_meta.json has a pose cache with at least a few frames of valid data.


In [ ]:
import h5py

def check_pose_cache(cache_path: Path) -> dict:
    """Check if a pose cache exists and has valid data."""
    result = {
        'exists': False,
        'has_data': False,
        'num_frames': 0,
        'error': None
    }
    
    if not cache_path.exists():
        result['error'] = 'Cache file does not exist'
        return result
    
    result['exists'] = True
    
    try:
        with h5py.File(cache_path, 'r') as f:
            # Count frames with pose data
            frame_keys = [k for k in f.keys() if k.startswith('frame_')]
            frames_with_data = 0
            frame_indices = []
            for frame_key in frame_keys:
                frame_group = f[frame_key]
                # Check if frame has at least one pose
                pose_keys = [k for k in frame_group.keys() if k.startswith('pose_')]
                if len(pose_keys) > 0:
                    # Verify pose has keypoints data
                    for pose_key in pose_keys:
                        pose_group = frame_group[pose_key]
                        if 'keypoints' in pose_group:
                            frames_with_data += 1
                            frame_indices.append(int(frame_key.split('_')[1]))
                            break
            result['frame_indices'] = frame_indices
            result['num_frames'] = frames_with_data
            result['has_data'] = frames_with_data > 0
            
            if frames_with_data == 0:
                result['error'] = 'Cache exists but has no frames with pose data'
    
    except Exception as e:
        result['error'] = f'Error reading cache: {str(e)}'
    
    return result

print("Pose cache checking function defined.")


In [ ]:
# Check pose caches for all videos in video_meta.json
# Use video_basename from original_path (stem of the video file)
pose_cache_results = []

for _, meta_row in video_meta_df.iterrows():
    original_path = meta_row.get('original_path')
    if pd.isna(original_path) or not original_path:
        print(f"Skipping video {meta_row.get('FileName')} because original_path is missing")
        continue
    
    video_basename = Path(original_path).stem
    pose_cache_path = pose_cache_base / video_basename / pose_cache_filename
    
    result = check_pose_cache(pose_cache_path)
    result['row_idx'] = meta_row.get('row_idx')
    result['FileName'] = meta_row.get('FileName')
    result['video_basename'] = video_basename
    result['cache_path'] = str(pose_cache_path)
    
    pose_cache_results.append(result)

pose_cache_df = pd.DataFrame(pose_cache_results)

print(f"Checked pose caches for {len(pose_cache_df)} videos")
print(f"\nPose cache status:")
print(f"  Caches that exist: {pose_cache_df['exists'].sum()} / {len(pose_cache_df)}")
print(f"  Caches with valid data: {pose_cache_df['has_data'].sum()} / {len(pose_cache_df)}")

# Show videos missing caches or with invalid data
missing_caches = pose_cache_df[~pose_cache_df['exists']]
invalid_caches = pose_cache_df[pose_cache_df['exists'] & ~pose_cache_df['has_data']]

if len(missing_caches) > 0:
    print(f"\n⚠️  Videos missing pose caches: {len(missing_caches)}")
    print(missing_caches[['row_idx', 'FileName', 'video_basename', 'error']].head(10))

if len(invalid_caches) > 0:
    print(f"\n⚠️  Videos with invalid/empty pose caches: {len(invalid_caches)}")
    print(invalid_caches[['row_idx', 'FileName', 'video_basename', 'num_frames', 'error']].head(10))

# Summary statistics
if pose_cache_df['has_data'].sum() > 0:
    valid_caches = pose_cache_df[pose_cache_df['has_data']]
    print(f"\nFrame count statistics (videos with valid pose data):")
    print(f"  Mean frames: {valid_caches['num_frames'].mean():.1f}")
    print(f"  Median frames: {valid_caches['num_frames'].median():.1f}")
    print(f"  Min frames: {valid_caches['num_frames'].min()}")
    print(f"  Max frames: {valid_caches['num_frames'].max()}")


In [ ]:
vid_pose_info = []
for _, meta_row in video_meta_df.iterrows():
    pose_info = {}
    original_path = meta_row.get('original_path')
    if pd.isna(original_path) or not original_path:
        print(f"Skipping video {meta_row.get('FileName')} because original_path is missing")
        continue

    altered_basename = meta_row['altered_basename']
    if not pd.isna(altered_basename):    
        print("ALTERED BASENAME: ", altered_basename)
        video_basename = altered_basename
    else:
        video_basename = Path(original_path).stem
    
    pose_cache_path = pose_cache_base / video_basename / pose_cache_filename
    
    source_file = meta_row['SourceFile']
    pose_info["row_idx"] = meta_row['row_idx']
    pose_info["source_file"] = source_file
    pose_info["cache_path"] = str(pose_cache_path)
    if pose_cache_path.exists():
        cache_info = check_pose_cache(pose_cache_path)
        pose_info['cache_info'] = cache_info
        pose_info['cache_info']['frame_indices'] = sorted(pose_info['cache_info']['frame_indices'])
        # if len(vid_pose_info) < 20:
        #     print(pose_info['cache_info']['frame_indices'])
        # print(f"Saved mask info for {meta_row['row_idx']}: {source_file}")
    else:
        print(f"  Cache does not exist for {source_file}")
        pose_info['cache_info'] = {}
    vid_pose_info.append(pose_info)

vid_pose_info_df = pd.DataFrame(vid_pose_info)
vid_pose_info_df


In [ ]:
import time

video_pose_path = Path('/orcd/data/satra/001/users/brukew/actreg/dataprep/video_pose_info.json')
video_pose_data = {"generated_at": time.strftime("%Y-%m-%dT%H:%M:%S.%fZ"), "count": len(vid_pose_info), "records": vid_pose_info}

with open(video_pose_path, 'w') as f:
    json.dump(video_pose_data, f, indent=2)

In [ ]:
missing_caches

In [ ]:
# clips_filename_stems = clips_df['filename'].apply(lambda x: Path(x).stem if pd.notna(x) else '')
missing_with_clips = missing_caches[missing_caches['FileName'].isin(clips_df['filename'])]
missing_with_clips


In [ ]:
missing_caches

In [ ]:
os.path.exists("/orcd/scratch/bcs/001/sensein/sails/cache_for_tracking/pose_sam3/7-18-17 (2)/dino-5scale_swin-l_8xb2-36e_coco_0.5_td-hm_hrnet-w48_dark-8xb32-210e_coco-wholebody-384x288_sam3guided.h5")

In [ ]:
affected_stems = affected_videos_df['source_file'].apply(lambda p: Path(p).stem)

no_pose_no_mask_cache = missing_caches[
    missing_caches['video_basename'].astype(str).isin(affected_stems)
]

no_pose_no_mask_cache

In [ ]:
print("20210912_214928" in str(affected_videos_df['source_file']))

In [ ]:
print(str(affected_videos_df['source_file']))

### Step 2: HRNet Keypoint Extraction Summary


In [ ]:
# Create summary
if 'pose_cache_df' in globals():
    pose_summary = {
        'Stage': 'HRNet Keypoint Extraction',
        'Source Script': 'actreg/dataprep/pose_gen/batch_sam_pose.py',
        'Total Videos in Metadata': len(video_meta_df),
        'Videos with Pose Caches': pose_cache_df['exists'].sum(),
        'Videos with Valid Pose Data': pose_cache_df['has_data'].sum(),
        'Videos Missing Caches': (~pose_cache_df['exists']).sum(),
        'Videos with Invalid/Empty Caches': (pose_cache_df['exists'] & ~pose_cache_df['has_data']).sum(),
    }
    
    print("=" * 80)
    print("HRNET KEYPOINT EXTRACTION SUMMARY")
    print("=" * 80)
    for key, value in pose_summary.items():
        print(f"{key}: {value}")
    
    missing_count = pose_summary['Videos Missing Caches']
    invalid_count = pose_summary['Videos with Invalid/Empty Caches']
    
    if missing_count == 0 and invalid_count == 0:
        print(f"\n✓ All videos have valid pose caches!")
    else:
        if missing_count > 0:
            print(f"\n⚠️  {missing_count} videos missing pose caches")
        if invalid_count > 0:
            print(f"⚠️  {invalid_count} videos have invalid/empty pose caches")
else:
    print("⚠️  Run previous cells to check pose caches first")


In [ ]:
# Create summary
if 'pose_cache_df' in globals():
    pose_summary = {
        'Stage': 'HRNet Keypoint Extraction',
        'Source Script': 'actreg/dataprep/pose_gen/batch_sam_pose.py',
        'Total Videos in Metadata': len(video_meta_df),
        'Videos with Pose Caches': pose_cache_df['exists'].sum(),
        'Videos with Valid Pose Data': pose_cache_df['has_data'].sum(),
        'Videos Missing Caches': (~pose_cache_df['exists']).sum(),
        'Videos with Invalid/Empty Caches': (pose_cache_df['exists'] & ~pose_cache_df['has_data']).sum(),
    }
    
    print("=" * 80)
    print("HRNET KEYPOINT EXTRACTION SUMMARY")
    print("=" * 80)
    for key, value in pose_summary.items():
        print(f"{key}: {value}")
    
    missing_count = pose_summary['Videos Missing Caches']
    invalid_count = pose_summary['Videos with Invalid/Empty Caches']
    
    if missing_count == 0 and invalid_count == 0:
        print(f"\n✓ All videos have valid pose caches!")
    else:
        if missing_count > 0:
            print(f"\n⚠️  {missing_count} videos missing pose caches")
        if invalid_count > 0:
            print(f"⚠️  {invalid_count} videos have invalid/empty pose caches")
else:
    print("⚠️  Run previous cells to check pose caches first")
